# 01 — Inspect COWC: scene inventory

This is the input to the split decision: pick `val_scenes` from the **colour** scenes (your UK target is RGB; Vaihingen is grey), favour RGB for training, and copy scene names verbatim so `validate_split_config` passes.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

# COWC scenes are huge; lift PIL's safety limit (same as the src modules do).
Image.MAX_IMAGE_PIXELS = None

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.annotations import find_negative_mask, count_points

RAW_ROOT = PROJECT_ROOT / "data/cowc/raw/cowc/datasets/ground_truth_sets"
META_DIR = PROJECT_ROOT / "data/cowc/metadata"
META_DIR.mkdir(parents=True, exist_ok=True)

print("Scanning:", RAW_ROOT)
print("Exists:", RAW_ROOT.exists())

Scanning: /Users/apple/Documents/carpark-project/data/cowc/raw/cowc/datasets/ground_truth_sets
Exists: True


### Discovery + per-scene measurement helpers

In [2]:
def discover_scenes(raw_root: Path) -> list[dict]:
    """Find every scene by its car-annotation mask, then locate the scene image.

    A scene is anchored by a *_Annotated_Cars.png file. The scene image is the
    same name with that suffix removed; the region is the parent folder.
    """
    scenes = []
    for car_mask in sorted(raw_root.rglob("*_Annotated_Cars.png")):
        stem = car_mask.name[: -len("_Annotated_Cars.png")]
        # the scene image is <stem> with a real image extension
        candidates = [car_mask.with_name(stem + ext)
                      for ext in (".png", ".tif", ".tiff", ".jpg", ".jpeg")]
        scene_img = next((c for c in candidates if c.exists()), None)
        scenes.append({
            "region": car_mask.parent.name,
            "scene": stem,
            "scene_image": scene_img,
            "car_mask": car_mask,
        })
    return scenes


def image_mode_and_size(path: Path):
    """(mode, width, height, is_color). is_color False for grayscale, or for
    RGB images whose channels are actually identical (grayscale stored as RGB)."""
    im = Image.open(path)
    mode = im.mode
    w, h = im.size                       # cheap: read from header, no full decode
    if mode in ("L", "LA", "I", "I;16", "1"):
        return mode, w, h, False
    thumb = im.convert("RGB")
    thumb.thumbnail((128, 128))          # forces one decode, kept small
    a = np.asarray(thumb)
    is_color = bool(np.any(a[..., 0] != a[..., 1]) or np.any(a[..., 1] != a[..., 2]))
    return mode, w, h, is_color


def build_inventory(raw_root: Path) -> pd.DataFrame:
    rows = []
    for s in discover_scenes(raw_root):
        rec = {"region": s["region"], "scene": s["scene"]}
        if s["scene_image"] is None:
            rec.update(mode="MISSING", width=None, height=None, is_color=None)
        else:
            mode, w, h, is_color = image_mode_and_size(s["scene_image"])
            rec.update(mode=mode, width=w, height=h, is_color=is_color)
        rec["n_cars"] = count_points(s["car_mask"])
        neg = find_negative_mask(s["scene_image"]) if s["scene_image"] else None
        rec["n_negatives"] = count_points(neg) if (neg and neg.exists()) else 0
        rows.append(rec)
        print(f"  {rec['region']:<22} {rec['scene'][:38]:<40} "
              f"{rec['mode']:<5} cars={rec['n_cars']:<6} neg={rec['n_negatives']}")
    return pd.DataFrame(rows)

### Build the inventory (one decode per scene — may take a minute on 32 big PNGs)

In [3]:
inv = build_inventory(RAW_ROOT)
inv = inv.sort_values(["is_color", "region", "scene"], ascending=[False, True, True])
out_csv = META_DIR / "cowc_scene_inventory.csv"
inv.to_csv(out_csv, index=False)
print(f"\nWrote {len(inv)} scenes to {out_csv}")
inv

  Columbus_CSUAV_AFRL    EO_Run01_s2_301_15_00_31.99319028-Oct-   RGB   cars=255    neg=549
  Columbus_CSUAV_AFRL    EO_Run01_s2_301_15_00_42.40561128-Oct-   RGB   cars=335    neg=824
  Columbus_CSUAV_AFRL    EO_Run01_s2_301_15_00_52.82681728-Oct-   RGBA  cars=1158   neg=840
  Potsdam_ISPRS          top_potsdam_2_10_RGB                     RGB   cars=102    neg=178
  Potsdam_ISPRS          top_potsdam_2_11_RGB                     RGBA  cars=99     neg=148
  Potsdam_ISPRS          top_potsdam_2_12_RGB                     RGBA  cars=114    neg=148
  Potsdam_ISPRS          top_potsdam_2_13_RGB                     RGBA  cars=140    neg=220
  Potsdam_ISPRS          top_potsdam_2_14_RGB                     RGBA  cars=79     neg=222
  Potsdam_ISPRS          top_potsdam_3_10_RGB                     RGBA  cars=174    neg=404
  Potsdam_ISPRS          top_potsdam_3_11_RGB                     RGBA  cars=174    neg=408
  Potsdam_ISPRS          top_potsdam_3_13_RGB                     RGBA  cars=273

,region,scene,mode,width,height,is_color,n_cars,n_negatives
3,Potsdam_ISPRS,top_potsdam_2_10_RGB,RGB,2220,2220,True,102,178
4,Potsdam_ISPRS,top_potsdam_2_11_RGB,RGBA,2220,2220,True,99,148
5,Potsdam_ISPRS,top_potsdam_2_12_RGB,RGBA,2220,2220,True,114,148
6,Potsdam_ISPRS,top_potsdam_2_13_RGB,RGBA,2220,2220,True,140,220
7,Potsdam_ISPRS,top_potsdam_2_14_RGB,RGBA,2220,2220,True,79,222
8,Potsdam_ISPRS,top_potsdam_3_10_RGB,RGBA,2220,2220,True,174,404
9,Potsdam_ISPRS,top_potsdam_3_11_RGB,RGBA,2220,2220,True,174,408
10,Potsdam_ISPRS,top_potsdam_3_13_RGB,RGBA,2220,2220,True,273,415
11,Potsdam_ISPRS,top_potsdam_4_10_RGB,RGBA,2220,2220,True,178,527
12,Potsdam_ISPRS,top_potsdam_5_10_RGB,RGB,2220,2220,True,163,474


### Summary for the split decision

In [4]:
# Quick read for the split decision
print("Scenes:", len(inv))
print("RGB (colour) scenes:", int((inv["is_color"] == True).sum()))
print("Grayscale scenes:   ", int((inv["is_color"] == False).sum()))
print("Total cars:", int(inv["n_cars"].sum()), " total negatives:", int(inv["n_negatives"].sum()))
print("\nBy region:")
print(inv.groupby("region").agg(scenes=("scene","count"),
                                cars=("n_cars","sum"),
                                colour=("is_color","max")))

Scenes: 32
RGB (colour) scenes: 28
Grayscale scenes:    4
Total cars: 37721  total negatives: 69551

By region:
                     scenes   cars  colour
region                                    
Columbus_CSUAV_AFRL       3   1748   False
Potsdam_ISPRS            13   2083    True
Selwyn_LINZ               3   1197    True
Toronto_ISPRS             3  10023    True
Utah_AGRC                 9  19807    True
Vaihingen_ISPRS           1   2863   False
